In [33]:
#current issue: does not include start/end time and start/end vert
# nested loop for each time going through all depths and then ANOTHER nested loop for all
# types?? seems excessive
# lat/lon format in dart

##current: removed type as input -> automatically adds obs for u_velocity and v_velocity
# HOW MANY POINTS IN A VERTICAL RANGE?
# hardcoded 50 points for depth, hardcoded obs_err_var
#are we generating obs_err_var??

In [34]:
import datetime as dt
import obs_to_obs_seq_in as obsin
import numpy as np

In [35]:
#given time range, depth range and frequency of obs, make list of obs
#so if 6-9months for 5min intervals, multiple obs for each time w/ varying depth
#waveglider location not changing
#so make class WAVEGLIDER, then six waveglider objects
#each object has fixed lat/lon/vert_unit, varying vert/time/obs_type?/err_var/metadata/external_FO



In [36]:
def interpolate_times(start: dt.datetime, end: dt.datetime, frequency):
    """given two times and a frequency of observations (in minutes), returns a list of times at given frequency"""
    times = []
    step = dt.timedelta(minutes=frequency)
    current_time = start
    while current_time <= end:
        times.append(current_time)
        current_time += step
    return times

In [ ]:
class MooredBuoy:
    """
    Initialize a moored buoy instrument at a fixed geographic location.

    Generates synthetic observation data over a vertical profile and time
    period, for converting into an ObsSequence file.

    Attributes:
        lat (float): Latitude of the moored instrument (degrees).
        lon (float): Longitude of the moored instrument (degrees).
        vert_unit (str): Unit of the vertical coordinate. Must be one of:
            ``'undefined'``, ``'surface (m)'``, ``'model level'``,
            ``'pressure (Pa)'``, ``'height (m)'``, ``'scale height'``.
        start_time (datetime.datetime): Start of the monitoring period.
        end_time (datetime.datetime): End of the monitoring period.
    
    Raises:
            ValueError: If ``vert_unit`` is not a recognised vertical unit.

    Example:
        .. code-block:: python

            buoy = MooredBuoy(
                lat=0.0,
                lon=140.8,
                vert_unit='surface (m)',
                start_time=dt.datetime(2023, 3, 2, 12, 30),
                end_time=dt.datetime(2023, 6, 5, 15, 30),
            )
    """
    vert = {
                    -2: "undefined",
                    -1: "surface (m)",
                    1: "model level",
                    2: "pressure (Pa)",
                    3: "height (m)",
                    4: "scale height",
                }
    #TODO: make this number variable
    vert_points = 50 #number of evenly spcaed points in given vertical range; currently set to 50
    
    def __init__(self, lat: float, lon: float, vert_unit: str, start_time: dt.datetime, end_time: dt.datetime):
        if vert_unit not in self.vert.values():
            raise ValueError(
                f"Invalid vert_unit '{vert_unit}'. "
                f"Must be one of: {sorted(self.vert.values())}"
            )
        self.lat = lat
        self.lon = lon
        self.vert_unit = vert_unit
        self.start_time = start_time
        self.end_time = end_time

    def data_generator(self, list_glider: list, vert_start: float, vert_end: float, obs_err_var: float, frequency, metadata = [] | None = None, external_FO = [] | None = None) -> list:
        """
        Populate ``obs_list`` with U and V velocity observations over a
        vertical profile and time series.

        Observations are generated at ``vertical_points`` evenly-spaced
        vertical levels between ``vert_start`` and ``vert_end``, at each
        timestep produced by ``interpolate_times``.

        Args:
            obs_list (list): Existing observation list to append to. Modified
                in-place and also returned.
            vert_start (float): Start of the vertical range, in units of
                ``self.vert_unit``.
            vert_end (float): End of the vertical range, in units of
                ``self.vert_unit``.
            obs_err_var (float): Observation error variance applied to all
                generated observations.
            frequency (int): Temporal sampling interval in minutes.
            metadata (list | None): Optional metadata attached to each
                observation. Defaults to an empty list.
            external_FO (list | None): Optional external forward-operator
                data. Defaults to an empty list.

        Returns:
            list: The updated ``obs_list`` with new observations appended.
        """
        vert_list = np.linspace(vert_start, vert_end, 50) #set to 50 points 
        time_list = interpolate_times(self.start_time, self.end_time, frequency) #frequency in mins
        for time in time_list:
            for vert in vert_list:
                obsin.add_obs_to_list(list_glider, self.lon, self.lat, float(vert), self.vert_unit, 'U_VELOCITY', time, obs_err_var, metadata, external_FO)
                obsin.add_obs_to_list(list_glider, self.lon, self.lat, float(vert), self.vert_unit, 'V_VELOCITY', time, obs_err_var, metadata, external_FO)
        return list_glider

In [38]:
obs_seq = obsin.create_obs_seq_in()
obs_list = obsin.create_obs_list()


In [39]:
wave1 = MooredBuoy(140.0, 0.0, 'height (m)',dt.datetime(2023,3,2,12), dt.datetime(2023,6,2,12))

In [40]:
wave1.data_generator(obs_list, 0, 4300, 0.001, 5)

[{'longitude': 140.0,
  'latitude': 0.0,
  'vertical': 0.0,
  'vert_unit': 'height (m)',
  'type': 'U_VELOCITY',
  'metadata': [],
  'external_FO': [],
  'seconds': 43200,
  'days': 154192,
  'time': '12:00:00',
  'obs_err_var': 0.001},
 {'longitude': 140.0,
  'latitude': 0.0,
  'vertical': 0.0,
  'vert_unit': 'height (m)',
  'type': 'V_VELOCITY',
  'metadata': [],
  'external_FO': [],
  'seconds': 43200,
  'days': 154192,
  'time': '12:00:00',
  'obs_err_var': 0.001},
 {'longitude': 140.0,
  'latitude': 0.0,
  'vertical': 87.75510204081633,
  'vert_unit': 'height (m)',
  'type': 'U_VELOCITY',
  'metadata': [],
  'external_FO': [],
  'seconds': 43200,
  'days': 154192,
  'time': '12:00:00',
  'obs_err_var': 0.001},
 {'longitude': 140.0,
  'latitude': 0.0,
  'vertical': 87.75510204081633,
  'vert_unit': 'height (m)',
  'type': 'V_VELOCITY',
  'metadata': [],
  'external_FO': [],
  'seconds': 43200,
  'days': 154192,
  'time': '12:00:00',
  'obs_err_var': 0.001},
 {'longitude': 140.0,
  

In [41]:
obsin.add_list_to_df(obs_list, obs_seq)

/Users/iranjan/fast-osse/obs_to_obs_seq_in.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  obs_seq.df = pd.concat([obs_seq.df, df_new], ignore_index=True)


In [42]:
print(obs_seq.df)


         obs_num                linked_list  longitude  latitude     vertical  \
0              1  -1          2          -1      140.0       0.0     0.000000   
1              2  1           3          -1      140.0       0.0     0.000000   
2              3  2           4          -1      140.0       0.0    87.755102   
3              4  3           5          -1      140.0       0.0    87.755102   
4              5  4           6          -1      140.0       0.0   175.510204   
...          ...                        ...        ...       ...          ...   
2649695  2649696  2649695     2649697    -1      140.0       0.0  4124.489796   
2649696  2649697  2649696     2649698    -1      140.0       0.0  4212.244898   
2649697  2649698  2649697     2649699    -1      140.0       0.0  4212.244898   
2649698  2649699  2649698     2649700    -1      140.0       0.0  4300.000000   
2649699  2649700  2649699     -1         -1      140.0       0.0  4300.000000   

          vert_unit        